# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mayurkusmade/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

**My lane: content refresh prioritization — ranking / scoring** (unchanged from ML-03).
The decision: *a content editor has ~50 slots a week — which pages should be refreshed first?*

Skills loaded for this card: `writing-data-contracts` + `flyrank/flyrank-data` (via `skills/README.md`).

This notebook is the write-down of **my lane's slice of the data river**, and every claim in it is
backed by a query that ran against the real release:

| Section | What it holds |
|---|---|
| 1 | the contract in plain words — one row, the tables, the window, the label, the exclusion |
| 2 | every field I touch, bucketed: feature / label / context / excluded |
| 3 | the proof — **exactly three verification queries** (grain, size + span, availability), then the five-feature frame (3.4) and the leakage trap (3.5) |
| 4 | what this slice can never tell me — one named limitation, measured |
| 5 | self-check |

**Data:** the gated release `FlyRank/internship-warehouse`, build
`flyrank_pseudonymized_warehouse_release_v20260703`, read with **DuckDB over `hf://`**. Only the two
month partitions this notebook names are ever fetched — never the 79M-row table.
Token: environment variable locally (`.env`, gitignored) or Colab Secret `HF_TOKEN` in Colab.
**Never paste a token into a cell** — this repo is public.

**Sealed month:** `fact_content_daily_performance_sample` is exactly June 2026 — the last month of
the panel, and therefore the natural outcome window of any past → future label. It is **never read
here**. Mid-panel month `2026-03` is where I iterate; June stays sealed for the later test pass.

In [1]:
# --- Setup: works in Colab (clones this repo) and locally (reads .env, walks up to the repo root) ---
import os
import subprocess
import sys
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/mayurkusmade/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"], check=True)
else:
    while not Path("data/raw").is_dir() and Path.cwd().parent != Path.cwd():
        os.chdir("..")


def hf_token():
    '''env var -> local .env (gitignored) -> Colab Secret -> interactive prompt. Never printed.'''
    token = os.environ.get("HF_TOKEN")
    if not token and Path(".env").exists():
        for line in Path(".env").read_text().splitlines():
            if line.strip().startswith("HF_TOKEN="):
                token = line.split("=", 1)[1].strip().strip('"').strip("'")
    if not token:
        try:
            from google.colab import userdata

            token = userdata.get("HF_TOKEN")
        except Exception:
            pass
    if not token:
        import getpass

        token = getpass.getpass("Hugging Face READ token (hf_...): ")
    return token


import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token()}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
BUILD = "flyrank_pseudonymized_warehouse_release_v20260703"


def month_ref(mm):
    '''One month partition of the daily fact table - the only big table this notebook touches.'''
    return f"read_parquet('{REL}/fact_content_daily_performance/month={mm}/data_0.parquet')"


DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"
FACT_MAR = month_ref("2026-03")   # feature month (mid-panel - iterate here)
FACT_APR = month_ref("2026-04")   # label month (the forward, observed outcome)

print(f"release       : {BUILD}")
print("read path     : DuckDB over hf:// - two month partitions, column-pruned")
print("feature month : 2026-03      label month: 2026-04      sealed: 2026-06")

release       : flyrank_pseudonymized_warehouse_release_v20260703
read path     : DuckDB over hf:// - two month partitions, column-pruned
feature month : 2026-03      label month: 2026-04      sealed: 2026-06


## 1. The contract, in plain words

**1) What one row means for my lane.** In the source table, one row = **one content item, for one
client, on one report date** — a page-day (`report_date` x `client_hash_id` x `content_hash_id`).
The row I *decide* on is different, and I state it separately: one row of my modelling frame =
**one content item for one client, summarised over one calendar month**. Ranking happens between
items, never between days.

**2) Which tables.** `fact_content_daily_performance` — partition `month=2026-03` for features and
`month=2026-04` for the label — plus `dim_clients` for per-client history coverage and access
profile. `dim_content` is available for metadata joins but these five features do not need it, and
`fact_content_query_90d` is excluded on purpose (answer 5).

**3) Which time window.** Feature window **2026-03-01 → 2026-03-31**, decision date **2026-03-31**,
label window **2026-04-01 → 2026-04-30**. The label month is genuinely *forward*: nothing about it is
knowable on the decision date. March is mid-panel on purpose, and **June 2026 — exactly the
`_sample` table — stays sealed**, so it is still usable as a test month later.

**4) What I predict or rank (label / proxy).** A **ranking** by the risk that next month's search
impressions fall at least 20% below the feature month's: `label = 1` when
`impressions(2026-04) < 0.8 x impressions(2026-03)`. The two impression totals are *observed* (GSC
measured them); the 20% band is *chosen* — and chosen to match the ±20% rule from ML-03 so the
numbers stay comparable. So the label is a **defined proxy over observed outcomes**, recorded as
such, not presented as a natural fact. The output for the editor is an ordered queue, measured at
the top (Precision@50 from ML-03).

**5) One thing I deliberately exclude.** All of `fact_content_query_90d`. Its window is a single
fixed 90-day block, and I measured that window as **2026-04-02 → 2026-06-30** — which *is* my label
month and beyond. Every `impressions_last30` / `*_90d` column in it would hand the model the answer
(section 4 prints that measurement). Also excluded: the engagement (GA4) fields for rows where
`ga4_data_available IS NOT TRUE` — those zeros mean "not tracked", not "no engagement".

In [2]:
# --- The contract as code, so later cells CHECK it instead of trusting the prose ---
LANE           = "content refresh prioritization (ranking / scoring)"
FEATURE_MONTH  = "2026-03"      # mid-panel: iterate here
LABEL_MONTH    = "2026-04"      # forward window: the observed outcome
SEALED_MONTH   = "2026-06"      # the _sample table - never read in this notebook

FEATURE_WINDOW = (date(2026, 3, 1), date(2026, 3, 31))
DECISION_DATE  = date(2026, 3, 31)
LABEL_WINDOW   = (date(2026, 4, 1), date(2026, 4, 30))

GRAIN          = ("client_hash_id", "content_hash_id", "report_date")   # source-table row
LANE_ROW       = "one content item x one client, summarised over 2026-03"
LABEL_RULE     = "label = 1 when impressions(2026-04) < 0.8 x impressions(2026-03)"
K_FOR_METRIC   = 50             # editor slots per week (ML-03)
FEATURES       = ["imp_mar", "ctr_mar", "pos_mar", "days_with_imp", "momentum_mar"]

# Window alignment is a claim like any other - so assert it rather than hoping.
assert FEATURE_WINDOW[1] < LABEL_WINDOW[0], "the feature window must close before the label window opens"
assert SEALED_MONTH not in (FEATURE_MONTH, LABEL_MONTH), "June must stay sealed"

contract = pd.DataFrame(
    [("row of the source table", " x ".join(GRAIN)),
     ("row of my modelling frame", LANE_ROW),
     ("tables", "fact_content_daily_performance (month=2026-03, month=2026-04) + dim_clients"),
     ("feature window", f"{FEATURE_WINDOW[0]} -> {FEATURE_WINDOW[1]}"),
     ("decision date", str(DECISION_DATE)),
     ("label window", f"{LABEL_WINDOW[0]} -> {LABEL_WINDOW[1]}  (forward, observed)"),
     ("label / proxy", LABEL_RULE),
     ("deliberately excluded", "fact_content_query_90d (its window overlaps the label month)"),
     ("sealed month", SEALED_MONTH)],
    columns=["contract answer", "value"])
display(contract)

,contract answer,value
0,row of the source table,client_hash_id x content_hash_id x report_date
1,row of my modelling frame,"one content item x one client, summarised over..."
2,tables,"fact_content_daily_performance (month=2026-03,..."
3,feature window,2026-03-01 -> 2026-03-31
4,decision date,2026-03-31
5,label window,"2026-04-01 -> 2026-04-30 (forward, observed)"
6,label / proxy,label = 1 when impressions(2026-04) < 0.8 x im...
7,deliberately excluded,fact_content_query_90d (its window overlaps th...
8,sealed month,2026-06


## 2. Fields: feature / label / context / excluded

Every field I plan to touch goes in exactly one bucket. IDs are pseudonyms — they join, group and
split, and are never learned from. The five features are the whole feature set on purpose; the
signal audit (ML-06) is where more get tested, not here.

In [3]:
# --- The inventory first: what columns actually exist in the two tables I use ---
fact_cols = con.sql(f"DESCRIBE SELECT * FROM {FACT_MAR}").df()[["column_name", "column_type"]]
dim_cols = con.sql(f"DESCRIBE SELECT * FROM {DIM_CLIENTS}").df()[["column_name", "column_type"]]
print(f"fact_content_daily_performance (2026-03 partition): {len(fact_cols)} columns")
print("  " + ", ".join(fact_cols.column_name))
print(f"dim_clients: {len(dim_cols)} columns")
print("  " + ", ".join(dim_cols.column_name))

buckets = pd.DataFrame([
 ("feature", "gsc_impressions (month total)", "-> imp_mar: measured inside the feature window"),
 ("feature", "gsc_clicks (month total)", "-> ctr_mar = clicks / impressions * 100, same window"),
 ("feature", "gsc_avg_position (month mean)", "-> pos_mar: 0 is the 'no position data' sentinel and is left out of the mean"),
 ("feature", "gsc_impressions > 0 (day count)", "-> days_with_imp: how many days in the month the page was seen at all"),
 ("feature", "gsc_impressions split at 2026-03-16", "-> momentum_mar = 2nd half / 1st half: within-month drift"),
 ("label", "SUM(gsc_impressions) over 2026-04", "the label's source is the FORWARD month - never a feature"),
 ("context", "client_hash_id, content_hash_id", "pseudonyms: join / group / split only, never learned from"),
 ("context", "report_date, month", "time keys used for grouping and windowing"),
 ("context", "gsc_data_available, ga4_data_available", "three-valued availability flags - used to FILTER"),
 ("context", "client_has_gsc, client_has_ga4", "client-level flags repeated on every row"),
 ("excluded", "fact_content_query_90d.*", "its only window is 2026-04-02 -> 2026-06-30, which contains my label month"),
 ("excluded", "ga4_* where ga4_data_available IS NOT TRUE", "zero-filled 'not tracked', not 'no engagement'"),
 ("excluded", "dim_content.last_optimized_date", "a product action, not an observation - baseline material, never a feature"),
 ("excluded", "2026-06 (the _sample table)", "sealed test month"),
], columns=["bucket", "field", "why it sits there"])
display(buckets)
print("No product decision fields (health_score / priority_score / action_type) exist in this release;")
print("if I ever rebuild one it is a baseline to beat, never a feature and never a label.")

fact_content_daily_performance (2026-03 partition): 31 columns
  report_date, client_hash_id, content_hash_id, client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available, gsc_impressions, gsc_clicks, gsc_sum_position, gsc_avg_position, ga4_pageviews, ga4_sessions, ga4_users, ga4_engaged_sessions, ga4_total_engagement_sec, sessions_organic, sessions_direct, sessions_referral, sessions_social, sessions_paid, sessions_ai, ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other, scroll_events, month
dim_clients: 9 columns
  client_hash_id, is_active, has_gsc_access, has_ga4_access, access_profile, client_created_date, client_updated_date, gsc_data_start, ga4_data_start


,bucket,field,why it sits there
0,feature,gsc_impressions (month total),-> imp_mar: measured inside the feature window
1,feature,gsc_clicks (month total),"-> ctr_mar = clicks / impressions * 100, same ..."
2,feature,gsc_avg_position (month mean),-> pos_mar: 0 is the 'no position data' sentin...
3,feature,gsc_impressions > 0 (day count),-> days_with_imp: how many days in the month t...
4,feature,gsc_impressions split at 2026-03-16,-> momentum_mar = 2nd half / 1st half: within-...
5,label,SUM(gsc_impressions) over 2026-04,the label's source is the FORWARD month - neve...
6,context,"client_hash_id, content_hash_id","pseudonyms: join / group / split only, never l..."
7,context,"report_date, month",time keys used for grouping and windowing
8,context,"gsc_data_available, ga4_data_available",three-valued availability flags - used to FILTER
9,context,"client_has_gsc, client_has_ga4",client-level flags repeated on every row


No product decision fields (health_score / priority_score / action_type) exist in this release;
if I ever rebuild one it is a baseline to beat, never a feature and never a label.


## 3. Prove it with queries

**Exactly three verification queries** — one per fact, each with its own output: the grain (3.1), my
slice's size and date span (3.2), and availability filtered with `IS TRUE` (3.3). Then **3.4 builds
the five-feature frame** and **3.5 springs the leakage trap**. Building and experimenting are not
verifying, which is why they are labelled separately.

In [4]:
# 3.1 - GRAIN. If the key is complete, no (client, content, date) appears twice.
dupes = con.sql(f'''
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS rows_for_this_key
    FROM {FACT_MAR}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
''').df()

rows, keys = con.sql(f'''
    SELECT COUNT(*) AS rows,
           COUNT(DISTINCT (client_hash_id, content_hash_id, report_date)) AS keys
    FROM {FACT_MAR}
''').fetchone()

print(f"2026-03 fact rows                  : {rows:,}")
print(f"distinct (client, content, date)   : {keys:,}")
print(f"keys that appear more than once    : {len(dupes)}   <- 0 rows returned means the grain holds")
dupes

2026-03 fact rows                  : 9,841,378
distinct (client, content, date)   : 9,841,378
keys that appear more than once    : 0   <- 0 rows returned means the grain holds


,client_hash_id,content_hash_id,report_date,rows_for_this_key


**3.2 Query 2 — the size and date span of my slice in that month.** An editor queue lives on
individual pages, so the counts that matter are pages and clients, not rows.

In [5]:
# 3.2 - SIZE AND SPAN.
slice_stats = con.sql(f'''
    SELECT COUNT(*)                    AS rows,
           MIN(report_date)            AS first_day,
           MAX(report_date)            AS last_day,
           COUNT(DISTINCT report_date) AS days_covered,
           COUNT(DISTINCT client_hash_id)  AS clients,
           COUNT(DISTINCT content_hash_id) AS content_items
    FROM {FACT_MAR}
''').df()
display(slice_stats)

,rows,first_day,last_day,days_covered,clients,content_items
0,9841378,2026-03-01,2026-03-31,31,55,331437


**3.3 Query 3 — availability, filtered with `IS TRUE`.** The availability flags are **three-valued**
(TRUE / FALSE / NULL), so `= FALSE` and `NOT ...` quietly mishandle NULLs. `IS TRUE` never does.
Everything built in 3.4 is restricted to rows where the search flag is true — the rows that fall out
are *not measured*, not "zero impressions".

In [6]:
# 3.3 - AVAILABILITY, with IS TRUE (three-valued flags).
avail = con.sql(f'''
    SELECT COUNT(*)                                              AS rows_all,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)     AS gsc_usable,
           COUNT(*) FILTER (WHERE gsc_data_available IS NOT TRUE) AS gsc_not_true,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)     AS ga4_usable,
           COUNT(*) FILTER (WHERE ga4_data_available IS NOT TRUE) AS ga4_not_true,
           COUNT(*) FILTER (WHERE ga4_data_available IS NULL)     AS ga4_null
    FROM {FACT_MAR}
''').df()
avail["gsc_usable_pct"] = (avail.gsc_usable / avail.rows_all * 100).round(1)
avail["ga4_usable_pct"] = (avail.ga4_usable / avail.rows_all * 100).round(1)
display(avail)

# The filter is only safe if it does not throw away real measurements: how many rows carry
# impressions yet sit OUTSIDE the flag?
outside = con.sql(f'''
    SELECT COUNT(*) FILTER (WHERE gsc_data_available IS TRUE     AND gsc_impressions > 0) AS positives_flagged_true,
           COUNT(*) FILTER (WHERE gsc_data_available IS NOT TRUE AND gsc_impressions > 0) AS positives_outside_flag,
           COUNT(*) FILTER (WHERE gsc_data_available IS NOT TRUE AND gsc_impressions IS NULL) AS nulls_outside_flag
    FROM {FACT_MAR}
''').df()
display(outside)
print("0 rows outside the flag carry impressions -> filtering with IS TRUE loses measurements, not data.")

,rows_all,gsc_usable,gsc_not_true,ga4_usable,ga4_not_true,ga4_null,gsc_usable_pct,ga4_usable_pct
0,9841378,3611061,6230317,413966,9427412,3018741,36.7,4.2


,positives_flagged_true,positives_outside_flag,nulls_outside_flag
0,3611061,0,0


0 rows outside the flag carry impressions -> filtering with IS TRUE loses measurements, not data.


### 3.4 Five features (max) — and when each one is knowable

Five, on purpose: one per question the editor's decision needs, and each one has to survive the
sentence **"knowable at the decision moment because …"**. If a feature cannot finish that sentence
using only 2026-03-01 → 2026-03-31, it does not belong in the frame.

Everything below is built with SQL inside DuckDB and only the ~100k-row result reaches pandas — the
point of reading the release this way.

In [7]:
# 3.4 - the five-feature frame. This is the ONE heavy scan in the notebook: it reads the 2026-03 and
#       2026-04 partitions once each. The result is cached to work/outputs/ (gitignored) because the
#       release explicitly asks you not to re-scan it repeatedly - repeated full scans hit HTTP 429.
CACHE = Path("work/outputs/w03_lane_frame_2026-03.parquet")
MIN_MONTH_IMPRESSIONS = 100   # below this a page is too thin to rank on: noise, not signal

if CACHE.exists():
    frame = pd.read_parquet(CACHE)
    origin = f"cache ({CACHE}) - delete the file to force a live scan"
else:
    frame = con.sql(f'''
        WITH feat AS (
            SELECT client_hash_id, content_hash_id,
                   SUM(gsc_impressions) AS imp_mar,
                   SUM(gsc_clicks)      AS clk_mar,
                   AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS pos_mar,
                   COUNT(*) FILTER (WHERE gsc_impressions > 0) AS days_with_imp,
                   SUM(CASE WHEN report_date >= DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS imp_h2,
                   SUM(CASE WHEN report_date <  DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS imp_h1
            FROM {FACT_MAR}
            WHERE gsc_data_available IS TRUE
            GROUP BY 1, 2
            HAVING SUM(gsc_impressions) >= {MIN_MONTH_IMPRESSIONS}
        ),
        label_window AS (
            SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_apr
            FROM {FACT_APR}
            WHERE gsc_data_available IS TRUE
            GROUP BY 1, 2
        )
        SELECT f.*, l.imp_apr
        FROM feat f
        LEFT JOIN label_window l
          ON f.client_hash_id = l.client_hash_id
         AND f.content_hash_id = l.content_hash_id
    ''').df()
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    frame.to_parquet(CACHE, index=False)
    origin = "live hf:// scan (2026-03 for features, 2026-04 for the label)"

frame["ctr_mar"] = 100 * frame.clk_mar / frame.imp_mar
frame["momentum_mar"] = (frame.imp_h2 / frame.imp_h1.replace(0, np.nan)).clip(0, 5)

kept = frame.dropna(subset=["imp_apr"]).copy()
kept["declined_apr"] = (kept.imp_apr < 0.8 * kept.imp_mar).astype(int)

print(f"frame source : {origin}")
print(f"items built (>= {MIN_MONTH_IMPRESSIONS} impressions in 2026-03) : {len(frame):,}")
print(f"items kept  (an April row exists to score against)    : {len(kept):,}"
      f"   ({len(frame) - len(kept):,} dropped - see section 4)")
print(f"clients in the frame : {frame.client_hash_id.nunique()}")

knowable = pd.DataFrame([
 ("imp_mar", "SUM(gsc_impressions) over 2026-03",
  "all 31 days it sums are already recorded when the decision is made"),
 ("ctr_mar", "clicks / impressions, both summed over 2026-03",
  "both counts come from the same closed window; a ratio needs no future data"),
 ("pos_mar", "mean gsc_avg_position over 2026-03, 0-sentinel excluded",
  "position is measured per day and averaged inside the window"),
 ("days_with_imp", "count of 2026-03 days with at least 1 impression",
  "a count of days, all of them inside the window"),
 ("momentum_mar", "2nd half of March / 1st half of March, clipped to [0, 5]",
  "both halves close on the decision date itself; it measures drift, not the outcome"),
], columns=["feature", "built from", "knowable at the decision moment because ..."])
display(knowable)

show = kept[["client_hash_id", "content_hash_id", "imp_mar", "ctr_mar", "pos_mar",
             "days_with_imp", "momentum_mar", "imp_apr", "declined_apr"]].head(5).copy()
show["client_hash_id"] = show.client_hash_id.str[:12] + "..."
show["content_hash_id"] = show.content_hash_id.str[:14] + "..."
display(show)
print(f"missing values in the frame: pos_mar {int(kept.pos_mar.isna().sum()):,}"
      f", momentum_mar {int(kept.momentum_mar.isna().sum()):,} (no first-half impressions)")
print(f"label base rate in this frame: {kept.declined_apr.mean():.4f}")

frame source : live hf:// scan (2026-03 for features, 2026-04 for the label)
items built (>= 100 impressions in 2026-03) : 101,441
items kept  (an April row exists to score against)    : 100,893   (548 dropped - see section 4)
clients in the frame : 44


,feature,built from,knowable at the decision moment because ...
0,imp_mar,SUM(gsc_impressions) over 2026-03,all 31 days it sums are already recorded when ...
1,ctr_mar,"clicks / impressions, both summed over 2026-03",both counts come from the same closed window; ...
2,pos_mar,"mean gsc_avg_position over 2026-03, 0-sentinel...",position is measured per day and averaged insi...
3,days_with_imp,count of 2026-03 days with at least 1 impression,"a count of days, all of them inside the window"
4,momentum_mar,"2nd half of March / 1st half of March, clipped...",both halves close on the decision date itself;...


,client_hash_id,content_hash_id,imp_mar,ctr_mar,pos_mar,days_with_imp,momentum_mar,imp_apr,declined_apr
0,client_73cda...,content_f956c0...,1119.0,0.178731,12.735874,31,1.103383,304.0,1
1,client_73cda...,content_f091a7...,621.0,0.000000,27.839946,30,0.789625,201.0,1
2,client_73cda...,content_8c5e18...,391.0,0.000000,27.904374,31,1.398773,319.0,0
3,client_73cda...,content_3baedd...,138.0,0.000000,22.180098,26,0.393939,55.0,1
4,client_73cda...,content_17c1e4...,862.0,0.000000,13.779266,31,1.215938,686.0,1


missing values in the frame: pos_mar 0, momentum_mar 4,759 (no first-half impressions)
label base rate in this frame: 0.5148


### 3.5 The trap: one label-derived column, added on purpose and deleted again

The leakage lesson from notebook 02, performed on real warehouse data. The quick score here is
deliberately cheap — a logistic regression on the five features, one 75/25 stratified split, seed 42.
It is **not** the lane's model (that is ML-08); it exists to be fooled.

The column I spring is `apr_change_pct = (imp_apr - imp_mar) / imp_mar`. It looks like an ordinary
measurement and it is one — measured in the **label month**, after the decision date. It is the
label's own source, rescaled. The model does not learn anything; it just reads the answer.

Two numbers are printed per run: **ROC-AUC** across the held-out items, and **Precision@50** — the
ML-03 metric. The leak drives both to the ceiling, while the honest row stays near 0.69 AUC and well
above the 0.53 base rate. (The honest numbers can move in the third decimal between runs; the leaked
score is exactly 1.0000, because that column *determines* the label.) When one column takes a score
to the ceiling, that column *is* the answer and the score has stopped measuring the model. The honest
row is the one that survives the deletion.

In [8]:
# 3.5 - the trap, performed and then removed.
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

model_data = kept.dropna(subset=FEATURES + ["declined_apr"]).copy()
print(f"rows with a complete feature set: {len(model_data):,}"
      f"   ({len(kept) - len(model_data):,} dropped for a missing feature)")
model_data["pos_mar"] = model_data.pos_mar.fillna(model_data.pos_mar.median())
model_data["momentum_mar"] = model_data.momentum_mar.fillna(1.0)


def quick_score(columns, tag):
    '''Cheap, reproducible, and only ever used to be fooled. Returns (ROC-AUC, Precision@50).'''
    X_train, X_test, y_train, y_test = train_test_split(
        model_data[columns], model_data.declined_apr,
        test_size=0.25, random_state=42, stratify=model_data.declined_apr)
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    top = pd.DataFrame({"y": y_test.values, "p": proba}).nlargest(K_FOR_METRIC, "p")
    print(f"  {tag:38} ROC-AUC {roc_auc_score(y_test, proba):.4f}   P@50 {top.y.mean():.3f}")
    return roc_auc_score(y_test, proba), top.y.mean()


print(f"held-out base rate (a random queue)      : {model_data.declined_apr.mean():.4f}")
honest_auc, honest_p50 = quick_score(FEATURES, "honest five features")

# --- the one column, sprung on purpose: the label's own source, rescaled ---
model_data["apr_change_pct"] = (model_data.imp_apr - model_data.imp_mar) / model_data.imp_mar * 100
leak_auc, leak_p50 = quick_score(FEATURES + ["apr_change_pct"], "+ apr_change_pct  <- LEAKED")

# --- delete it and keep the honest number ---
model_data = model_data.drop(columns=["apr_change_pct"])
assert "apr_change_pct" not in model_data.columns, "the leaked column must be gone"
honest_again_auc, honest_again_p50 = quick_score(FEATURES, "leak removed, honest again")

print(f"leak column present after deletion: {'apr_change_pct' in model_data.columns}")
display(pd.DataFrame([
 ("honest, five features", round(honest_auc, 4), round(honest_p50, 3), "kept"),
 ("five features + apr_change_pct", round(leak_auc, 4), round(leak_p50, 3), "DELETED - label-derived"),
 ("honest again, leak removed", round(honest_again_auc, 4), round(honest_again_p50, 3), "kept"),
], columns=["quick score", "ROC-AUC (held-out)", "Precision@50", "status"]))
print("Lesson: a feature that lifts a score to perfection is usually the label wearing a different name.")

rows with a complete feature set: 96,134   (4,759 dropped for a missing feature)
held-out base rate (a random queue)      : 0.5324


  honest five features                   ROC-AUC 0.6921   P@50 0.820


  + apr_change_pct  <- LEAKED            ROC-AUC 1.0000   P@50 1.000


  leak removed, honest again             ROC-AUC 0.6921   P@50 0.820
leak column present after deletion: False


,quick score,ROC-AUC (held-out),Precision@50,status
0,"honest, five features",0.6921,0.82,kept
1,five features + apr_change_pct,1.0000,1.00,DELETED - label-derived
2,"honest again, leak removed",0.6921,0.82,kept


Lesson: a feature that lifts a score to perfection is usually the label wearing a different name.


## 4. What this slice can never tell me

**Named limitation: my lane's slice is a young, thin slice of an unbalanced panel — 44 of the 104
pseudonymized clients, whose GSC history is wildly unequal.** One of those clients has four days of
history before the decision date; the oldest has fourteen months. A frame that pools their items
together is a frame in which the *same* feature means different things per client, so "declining"
is not measured over a comparable baseline. Everything I claim from this frame is a statement about
those 44 clients in that month — not about the release, and not about a client whose tracking
started later.

The other limits, all measured below:

- **The frame silently drops the biggest declines.** 548 items have no April row at all, so the frame
  keeps only the 100,893 items it can score against. "Traffic fell to zero" and "the item was removed
  or unpublished" are indistinguishable here — and that is exactly where the most extreme case lives.
- **Engagement is barely knowable.** Only ~4% of March rows carry `ga4_data_available IS TRUE`, so any
  GA4-based feature would be knowable for a sliver of the slice. That is why the five features are all
  search-side.
- **The ±20% band is a chosen rule, not a fact.** A client whose whole inventory fell labels nearly
  everything; a healthy client labels almost nothing. A random split mixes clients and flatters the
  score — ML-08 must group the split by `client_hash_id`.
- **Position is not comparable across clients** (different query mixes), and `gsc_avg_position = 0`
  means "no position data", not rank zero.
- **Nothing here is causal.** This is decision-support for an editor's queue on anonymized historical
  data. It is not a prediction of Google's algorithm and not evidence that refreshing a page recovers
  traffic.

In [9]:
# 4 - the limits, measured rather than asserted.
clients = con.sql(f"SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start FROM {DIM_CLIENTS}").df()
in_frame = clients[clients.client_hash_id.isin(frame.client_hash_id.unique())].copy()
history_days = (pd.Timestamp(DECISION_DATE) - in_frame.gsc_data_start).dt.days

print(f"clients in dim_clients                  : {len(clients)}")
print(f"clients my 2026-03 frame actually covers: {len(in_frame)}  ({len(in_frame) / len(clients):.1%})")
print(f"their GSC history starts between        : {in_frame.gsc_data_start.min().date()}"
      f" and {in_frame.gsc_data_start.max().date()}")
print(f"history at the decision date (days)     : median {history_days.median():.0f},"
      f" min {history_days.min():.0f}, max {history_days.max():.0f}")
print(f"items dropped for having no April row   : {int(frame.imp_apr.isna().sum())} of {len(frame):,}"
      f"  ({frame.imp_apr.isna().mean():.1%}) - the largest declines are the ones I cannot see")
print()
print("access profiles inside the frame:")
print(in_frame.access_profile.value_counts().to_string())

q90 = con.sql(f'''
    SELECT COUNT(*) AS rows,
           MIN(window_start) AS window_start,
           MAX(window_start) AS window_start_max,
           MIN(window_end)   AS window_end,
           MAX(window_end)   AS window_end_max
    FROM read_parquet('{REL}/fact_content_query_90d.parquet')
''').df()
print("\nthe excluded query table, measured - its window is my label month and beyond:")
display(q90)

clients in dim_clients                  : 104
clients my 2026-03 frame actually covers: 44  (42.3%)
their GSC history starts between        : 2025-01-27 and 2026-03-27
history at the decision date (days)     : median 166, min 4, max 428
items dropped for having no April row   : 548 of 101,441  (0.5%) - the largest declines are the ones I cannot see

access profiles inside the frame:
access_profile
gsc_and_ga4                      37
gsc_only                          6
no_search_or_analytics_access     1



the excluded query table, measured - its window is my label month and beyond:


,rows,window_start,window_start_max,window_end,window_end_max
0,2414248,2026-04-02,2026-04-02,2026-06-30,2026-06-30


## 5. Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (`Runtime -> Run all`; executed locally with `nbclient`, outputs saved in this file)
- [x] The three verification queries are 3.1 (grain), 3.2 (size + span) and 3.3 (availability with `IS TRUE`), each showing its own output
- [x] Five features, each with a "knowable at the decision moment because …" line; nothing from the label month is a feature
- [x] The label-derived column was added, shown to push ROC-AUC to 1.0000, then deleted — the honest number is kept
- [x] One named limitation of my slice, measured in section 4
- [x] No client names, URLs, or raw queries anywhere — IDs are pseudonyms, shown truncated, used for grouping only
- [x] Careful words throughout: observed, measured, directional, decision-support
- [x] June 2026 (the `_sample` table) never read — it stays sealed as the test month

**Carried into ML-07 / ML-08:** the ±20% proxy needs a real forward-window definition at scale, the
split must group by `client_hash_id`, the feature frame should extend back over more than one month
(seasonality never enters a single-month frame), and Precision@50's K should come from real editor
capacity rather than default habit.